# Authentication Workshop: Sessions & Cookies from Scratch

## What we'll build

A login system for our AgentFlow chatbot. By the end, you'll understand:

1. **Where users are stored** → SQLite database
2. **How the server remembers you** → Session tokens in the DB  
3. **How the browser proves it's you** → Cookies sent on every request
4. **How protected routes work** → Gatekeeper checks the cookie before allowing access
5. **How logout works** → Delete from BOTH sides (server DB + browser cookie)

---

**Key insight:** HTTP is stateless. The server forgets you after every request. Sessions + cookies are how we fake "memory" on top of a forgetful protocol.

```
Request 1: "Show me my dashboard"  →  Server: "Who are you??"
Request 2: "Show me my dashboard"  →  Server: "Who are you??" (again!)
```

We need something that says **"I already proved who I am"** on every request. That's a cookie.

---
# STEP 1: Connect to the Database

Before we can store users, we need a database. We'll use **SQLite** — a file-based database that requires zero setup. SQLAlchemy lets us talk to it using Python instead of raw SQL.

```
Our app  ←→  SQLAlchemy (ORM)  ←→  SQLite file (agentflow_workshop.db)
```

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, DateTime, Boolean
from sqlalchemy.orm import sessionmaker, DeclarativeBase
import os

# Delete old workshop DB if it exists (fresh start every time)
if os.path.exists("agentflow_workshop.db"):
    os.remove("agentflow_workshop.db")
    print("Deleted old database — starting fresh\n")

# Create the database connection
engine = create_engine("sqlite:///agentflow_workshop.db", connect_args={"check_same_thread": False})

# Session factory — creates a new DB connection when we call it
SessionLocal = sessionmaker(bind=engine)

# Base class — every table inherits from this
class Base(DeclarativeBase):
    pass

print("Database engine created: agentflow_workshop.db")
print(f"Connection string: sqlite:///agentflow_workshop.db")

---
# STEP 2: Define the Tables (Models)

We need two tables:

| Table | Purpose |
|-------|---------|
| **users** | Stores WHO exists (email, name) |
| **sessions** | Stores WHO is currently logged in (token → user_id) |

```
users table:                    sessions table:
┌────┬─────────────┬───────┐    ┌────┬──────────────────┬─────────┬────────────┐
│ id │ email       │ name  │    │ id │ session_token    │ user_id │ expires_at │
├────┼─────────────┼───────┤    ├────┼──────────────────┼─────────┼────────────┤
│ 1  │ alice@test  │ Alice │    │ 1  │ a6f3b9c2d8e4...  │ 1       │ tomorrow   │
│ 2  │ bob@test    │ Bob   │    │ 2  │ 7d2e1f8a9b3c...  │ 1       │ tomorrow   │
└────┴─────────────┴───────┘    └────┴──────────────────┴─────────┴────────────┘
                                 ↑ Alice has TWO sessions (phone + laptop)
```

In [ ]:
from datetime import datetime, timezone

# ── Table 1: Users ──────────────────────────────────
class User(Base):
    __tablename__ = "users"
    
    id = Column(Integer, primary_key=True, index=True)
    email = Column(String, unique=True, nullable=False)   # No two users with same email
    name = Column(String, nullable=False)
    picture = Column(String, nullable=True)               # Profile pic URL (for Google later)
    active = Column(Boolean, default=True)                 # Can be deactivated
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))


# ── Table 2: Sessions ──────────────────────────────
class SessionRecord(Base):
    __tablename__ = "sessions"
    
    id = Column(Integer, primary_key=True, index=True)
    session_token = Column(String, unique=True, nullable=False, index=True)  # The cookie value
    user_id = Column(Integer, nullable=False)              # Which user this session belongs to
    expires_at = Column(DateTime, nullable=False)          # When this session dies


# Create the actual tables in the database file
Base.metadata.create_all(bind=engine)

print("Tables created!")
print(f"Tables in database: {list(Base.metadata.tables.keys())}")

In [ ]:
# Let's verify — the tables exist but are empty
import sqlite3

conn = sqlite3.connect("agentflow_workshop.db")
cursor = conn.cursor()

# Show all tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Tables in database:")
for t in tables:
    print(f"  - {t[0]}")

# Show they're empty
print(f"\nUsers count:    {cursor.execute('SELECT COUNT(*) FROM users').fetchone()[0]}")
print(f"Sessions count: {cursor.execute('SELECT COUNT(*) FROM sessions').fetchone()[0]}")
conn.close()

---
# STEP 3: Signup — Create a User

Signup = **INSERT a row into the users table**. That's it. No session, no cookie, no login.

The user *exists* but is NOT *logged in* yet.

```
What signup does:
  INSERT INTO users (email, name) VALUES ('alice@test.com', 'Alice')

What signup does NOT do:
  ✗ Create a session
  ✗ Set a cookie
  ✗ Log the user in
```

In [ ]:
# ── SIGNUP: Create users ──────────────────────────

db = SessionLocal()

# Create Alice
alice = User(email="alice@test.com", name="Alice")
db.add(alice)
db.commit()
db.refresh(alice)    # reload from DB to get the auto-generated id
print(f"Created user: id={alice.id}, email={alice.email}, name={alice.name}")

# Create Bob  
bob = User(email="bob@test.com", name="Bob")
db.add(bob)
db.commit()
db.refresh(bob)
print(f"Created user: id={bob.id}, email={bob.email}, name={bob.name}")

db.close()

In [ ]:
# ── INSPECT: What's in the database now? ──────────

conn = sqlite3.connect("agentflow_workshop.db")
cursor = conn.cursor()

print("=== USERS TABLE ===")
print(f"{'id':<5} {'email':<20} {'name':<10} {'active':<8} {'created_at'}")
print("-" * 70)
for row in cursor.execute("SELECT id, email, name, active, created_at FROM users"):
    print(f"{row[0]:<5} {row[1]:<20} {row[2]:<10} {row[3]:<8} {row[4]}")

print(f"\n=== SESSIONS TABLE ===")
count = cursor.execute("SELECT COUNT(*) FROM sessions").fetchone()[0]
print(f"Sessions: {count}  (empty — nobody has logged in yet!)")

conn.close()

---
# STEP 4: Login — The Most Important Step

Login does **TWO things**:

1. **Server side:** Generate a random token → store it in the `sessions` table (linked to user_id)
2. **Browser side:** Set that token as a cookie on the HTTP response

Think of it like a **concert wristband**:
- The venue (server) gives you a wristband (cookie) at the door (login)  
- You show the wristband on every entry (every HTTP request)  
- The venue checks it against their guest list (sessions table)
- When you leave (logout), they cut the wristband and cross you off the list

```
LOGIN FLOW:

  1. User sends email: "alice@test.com"
              │
  2. Server:  SELECT * FROM users WHERE email = 'alice@test.com'
              → Found! id = 1
              │
  3. Server:  token = secrets.token_hex(32)  →  "a6f3b9c2d8e4f1..."
              │
  4. Server:  INSERT INTO sessions (token, user_id, expires_at)
              VALUES ('a6f3b9c2d8e4f1...', 1, '2026-04-25')
              │
  5. Server:  HTTP Response headers:
              Set-Cookie: session_token=a6f3b9c2d8e4f1...; HttpOnly; SameSite=Lax
              │
  6. Browser: "Got it, I'll save this cookie and send it on every future request"
```

In [ ]:
# ── LOGIN: Simulate what the server does ──────────

import secrets
from datetime import timedelta

db = SessionLocal()

# === Step 1: Find the user ===
login_email = "alice@test.com"
user = db.query(User).filter(User.email == login_email).first()

if not user:
    print(f"ERROR: User '{login_email}' not found!")
else:
    print(f"Step 1 - Found user: id={user.id}, name={user.name}")
    
    # === Step 2: Generate a random token ===
    token = secrets.token_hex(32)   # 64-character random hex string
    print(f"Step 2 - Generated token: {token}")
    print(f"         Token length: {len(token)} characters (practically unguessable)")
    
    # === Step 3: Store the session in the database ===
    session = SessionRecord(
        session_token=token,
        user_id=user.id,
        expires_at=datetime.now(timezone.utc) + timedelta(hours=24),
    )
    db.add(session)
    db.commit()
    print(f"Step 3 - Session saved to DB: user_id={session.user_id}, expires={session.expires_at}")
    
    # === Step 4: This is what the cookie would be ===
    print(f"\nStep 4 - The HTTP response would include:")
    print(f"         Set-Cookie: session_token={token}; HttpOnly; SameSite=Lax; Max-Age=86400")
    
    # Save the token so we can use it in later cells
    alice_token = token
    print(f"\n(Saved token as 'alice_token' for next steps)")

db.close()

In [ ]:
# ── INSPECT: Now there's a session in the DB! ────

conn = sqlite3.connect("agentflow_workshop.db")
cursor = conn.cursor()

print("=== USERS TABLE ===")
print(f"{'id':<5} {'email':<20} {'name':<10}")
print("-" * 40)
for row in cursor.execute("SELECT id, email, name FROM users"):
    print(f"{row[0]:<5} {row[1]:<20} {row[2]:<10}")

print(f"\n=== SESSIONS TABLE ===")
print(f"{'id':<5} {'token (first 20 chars)':<25} {'user_id':<10} {'expires_at'}")
print("-" * 70)
for row in cursor.execute("SELECT id, session_token, user_id, expires_at FROM sessions"):
    print(f"{row[0]:<5} {row[1][:20]}...  {row[2]:<10} {row[3]}")

print(f"\n^ The session_token in the DB = the cookie value in the browser")
print(f"  That's how they're connected!")

conn.close()

---
# STEP 5: Reading the Session — "Who is this?"

Every time the browser makes a request, it sends the cookie automatically:

```
Browser → Server:
    GET /auth/me
    Cookie: session_token=a6f3b9c2d8e4f1...
```

The server reads the cookie and looks it up:

```
  1. Read cookie value from request     → "a6f3b9c2d8e4f1..."
  2. SELECT * FROM sessions WHERE token = "a6f3b9c2d8e4f1..."
     → Found! user_id = 1, expires = tomorrow
  3. Is it expired? No ✓
  4. SELECT * FROM users WHERE id = 1
     → Alice!
  5. Return: {"name": "Alice", "email": "alice@test.com"}
```

**The server doesn't "remember" you. It LOOKS YOU UP every single time.**

In [ ]:
# ── SIMULATE: /auth/me — What happens when the browser sends the cookie ──

db = SessionLocal()

# Simulate: the browser sent this cookie
incoming_cookie = alice_token   # In reality, this comes from request.cookies.get("session_token")

print(f"=== Incoming request ===")
print(f"Cookie value: {incoming_cookie[:20]}...\n")

# Step 1: Look up the token in the sessions table
session = db.query(SessionRecord).filter(SessionRecord.session_token == incoming_cookie).first()

if not session:
    print("RESULT: 401 Unauthorized — token not found in database")
else:
    print(f"Step 1 - Session found! user_id={session.user_id}, expires={session.expires_at}")
    
    # Step 2: Check if expired
    if session.expires_at < datetime.utcnow():
        print("RESULT: 401 Unauthorized — session expired")
    else:
        print(f"Step 2 - Not expired ✓")
        
        # Step 3: Get the user
        user = db.query(User).filter(User.id == session.user_id).first()
        print(f"Step 3 - Found user: {user.name} ({user.email})")
        
        print(f"\n=== Response ===")
        print(f'{{"id": {user.id}, "email": "{user.email}", "name": "{user.name}"}}')

db.close()

---
# STEP 6: Fake Cookie — Prove You Can't Forge Sessions

What if an attacker tries to make up a cookie value? Let's try it.

In [ ]:
# ── FAKE COOKIE: What happens with a made-up token? ──

db = SessionLocal()

fake_cookie = "i_am_a_hacker_trying_to_break_in_12345"

print(f"=== Attacker's request ===")
print(f"Cookie value: {fake_cookie}\n")

# Look it up in the database
session = db.query(SessionRecord).filter(SessionRecord.session_token == fake_cookie).first()

if not session:
    print("RESULT: 401 Unauthorized — token not found in database!")
    print("")
    print("Why this fails:")
    print(f"  - The real token is 64 hex characters (e.g., 'a6f3b9c2d8e4...')")
    print(f"  - There are 16^64 possible tokens = 10^77 combinations")
    print(f"  - That's more than atoms in the universe")
    print(f"  - You CANNOT guess a valid token")

db.close()

---
# STEP 7: Multiple Sessions — One User, Two Devices

When Alice logs in from her phone AND her laptop, she gets **two different sessions**. Each has its own token. Logging out on one doesn't kill the other.

```
sessions table after two logins:
┌────┬──────────────────────┬─────────┐
│ id │ session_token        │ user_id │
├────┼──────────────────────┼─────────┤
│ 1  │ a6f3b9c2d8e4... (A) │ 1       │  ← laptop
│ 2  │ 7d2e1f8a9b3c... (B) │ 1       │  ← phone
└────┴──────────────────────┴─────────┘
Both tokens map to Alice (user_id=1), but they're independent.
```

In [ ]:
# ── MULTIPLE SESSIONS: Alice logs in from a second device ──

db = SessionLocal()

# Second login for Alice (simulates phone login)
user = db.query(User).filter(User.email == "alice@test.com").first()
second_token = secrets.token_hex(32)

session2 = SessionRecord(
    session_token=second_token,
    user_id=user.id,
    expires_at=datetime.now(timezone.utc) + timedelta(hours=24),
)
db.add(session2)
db.commit()

alice_token_2 = second_token

# Show all sessions
print("=== ALL SESSIONS IN DATABASE ===")
print(f"{'id':<5} {'token (first 16)':<20} {'user_id':<10} {'device'}")
print("-" * 55)
all_sessions = db.query(SessionRecord).all()
for i, s in enumerate(all_sessions):
    device = "laptop" if i == 0 else "phone"
    print(f"{s.id:<5} {s.session_token[:16]}...  {s.user_id:<10} {device}")

print(f"\nBoth sessions belong to Alice (user_id=1)")
print(f"Each has a DIFFERENT token — they're independent")

db.close()

In [ ]:
# ── VERIFY: Both tokens work independently ──

db = SessionLocal()

print("=== Testing token A (laptop) ===")
s1 = db.query(SessionRecord).filter(SessionRecord.session_token == alice_token).first()
u1 = db.query(User).filter(User.id == s1.user_id).first()
print(f"  Token: {alice_token[:16]}...")
print(f"  Result: {u1.name} ({u1.email}) ✓")

print(f"\n=== Testing token B (phone) ===")
s2 = db.query(SessionRecord).filter(SessionRecord.session_token == alice_token_2).first()
u2 = db.query(User).filter(User.id == s2.user_id).first()
print(f"  Token: {alice_token_2[:16]}...")
print(f"  Result: {u2.name} ({u2.email}) ✓")

print(f"\nBoth work! Same user, different wristbands.")

db.close()

---
# STEP 8: Logout — Delete from BOTH Sides

Logout does **TWO deletes**:

```
1. Server:  DELETE FROM sessions WHERE token = 'a6f3b9c2d8e4...'
            (server forgets this session)

2. Browser: Set-Cookie: session_token=; Max-Age=0
            (browser deletes the cookie)
```

Even if someone stole the old cookie value BEFORE logout, it's useless now — the server-side record is gone. The wristband has been cut AND crossed off the guest list.

In [ ]:
# ── LOGOUT: Delete session A (laptop) ──────────────

db = SessionLocal()

logout_token = alice_token   # The laptop session

# Step 1: Find and delete the session
session = db.query(SessionRecord).filter(SessionRecord.session_token == logout_token).first()
if session:
    db.delete(session)
    db.commit()
    print(f"Step 1 - Deleted session from DB (token: {logout_token[:16]}...)")
    print(f"Step 2 - HTTP response would include: Set-Cookie: session_token=; Max-Age=0")
else:
    print("Session not found (already logged out?)")

# Step 2: Verify — try to use the old token
print(f"\n=== Trying the old token after logout ===")
session_check = db.query(SessionRecord).filter(SessionRecord.session_token == logout_token).first()
if not session_check:
    print(f"Token {logout_token[:16]}... → NOT FOUND in DB")
    print(f"RESULT: 401 Unauthorized — even if someone stole this cookie, it's dead")

# Step 3: But the OTHER session (phone) still works!
print(f"\n=== Token B (phone) still works? ===")
session_b = db.query(SessionRecord).filter(SessionRecord.session_token == alice_token_2).first()
if session_b:
    user = db.query(User).filter(User.id == session_b.user_id).first()
    print(f"Token {alice_token_2[:16]}... → {user.name} ✓")
    print(f"Logging out on laptop did NOT kill the phone session!")

db.close()

---
# STEP 9: Cookie Security Flags

When setting the cookie, we use special flags to prevent attacks:

| Flag | What it does | Protects against |
|------|-------------|-----------------|
| **`HttpOnly`** | JavaScript can't read the cookie | **XSS** — attacker injects `<script>` that tries `document.cookie` |
| **`SameSite=Lax`** | Cookie only sent on same-site requests | **CSRF** — attacker tricks your browser into making requests to our server |
| **`Secure`** | Cookie only sent over HTTPS | **Man-in-the-middle** — attacker sniffing network traffic |
| **`Max-Age`** | Cookie expires after N seconds | **Stale sessions** — old cookies hanging around forever |

```python
response.set_cookie(
    key="session_token",
    value=token,
    httponly=True,       # JS can't access — document.cookie returns ""
    samesite="lax",      # Only same-site requests
    max_age=86400,       # 24 hours in seconds
)
```

### The HttpOnly test you can do in the browser:

```
1. Login through the frontend
2. Open DevTools → Application → Cookies → you SEE the cookie
3. Open Console → type: document.cookie → returns "" (empty!)
4. The cookie EXISTS but JavaScript CANNOT read it
5. If an attacker injects JS via XSS, they can't steal the cookie
```

---
# STEP 10: The Gatekeeper — Protecting Routes

In FastAPI, we use `Depends()` to run a function BEFORE the route handler. If the function raises an error, the route never executes.

```python
# This function runs on EVERY request to a protected route
def get_current_user(request, db):
    token = request.cookies.get("session_token")    # 1. Read cookie
    session = db.query(SessionRecord).filter(...)    # 2. Lookup in DB
    if not session: raise 401                        # 3. Reject if invalid
    user = db.query(User).filter(...)                # 4. Get the user
    return user                                      # 5. Pass to route

# Usage — the chat route is now protected:
@app.post("/chat")
def chat(question, user = Depends(get_current_user)):
    # This code ONLY runs if get_current_user succeeded
    # If no cookie / invalid session → 401 returned, chat() never executes
    return ask(question)
```

```
Request without cookie:
  Browser → /chat → get_current_user() → NO COOKIE → 401 STOP ✗
                                          (chat() never runs)

Request with valid cookie:
  Browser → /chat → get_current_user() → cookie found → session found → user found
                                          → chat() runs → answer returned ✓
```

In [ ]:
# ── SIMULATE: The gatekeeper in action ──────────────

db = SessionLocal()

def simulate_get_current_user(cookie_value):
    """This is exactly what auth/dependencies.py does"""
    
    print(f"  1. Reading cookie... ", end="")
    if not cookie_value:
        print("NO COOKIE")
        return None, "401: Not logged in — no session cookie"
    print(f"'{cookie_value[:16]}...'")
    
    print(f"  2. Looking up in sessions table... ", end="")
    session = db.query(SessionRecord).filter(SessionRecord.session_token == cookie_value).first()
    if not session:
        print("NOT FOUND")
        return None, "401: Invalid session — token not in database"
    print(f"found! user_id={session.user_id}")
    
    print(f"  3. Checking expiry... ", end="")
    if session.expires_at < datetime.utcnow():
        print("EXPIRED")
        return None, "401: Session expired"
    print(f"valid until {session.expires_at}")
    
    print(f"  4. Getting user... ", end="")
    user = db.query(User).filter(User.id == session.user_id).first()
    if not user:
        print("NOT FOUND")
        return None, "401: User not found"
    print(f"{user.name} ({user.email})")
    
    return user, None


# === Test 1: No cookie ===
print("=" * 50)
print("TEST 1: Request with NO cookie")
print("=" * 50)
user, err = simulate_get_current_user(None)
print(f"  RESULT: {err}\n")

# === Test 2: Fake cookie ===
print("=" * 50)
print("TEST 2: Request with FAKE cookie")
print("=" * 50)
user, err = simulate_get_current_user("totally_fake_token")
print(f"  RESULT: {err}\n")

# === Test 3: Valid cookie ===
print("=" * 50)
print("TEST 3: Request with VALID cookie")
print("=" * 50)
user, err = simulate_get_current_user(alice_token_2)
if user:
    print(f"  RESULT: ACCESS GRANTED — welcome {user.name}!")
    print(f"  → Now the /chat endpoint would run")

db.close()

---
# STEP 11: Full Picture — The Complete Session Lifecycle

```
┌─────────────────────────────────────────────────────────────────┐
│                    SESSION LIFECYCLE                             │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  SIGNUP (one time)                                              │
│  ├─ INSERT INTO users (email, name)                            │
│  └─ No session, no cookie                                       │
│                                                                 │
│  LOGIN (creates session)                                        │
│  ├─ Find user in DB                                             │
│  ├─ Generate random token                                       │
│  ├─ INSERT INTO sessions (token, user_id, expires)             │
│  └─ Set-Cookie: session_token=<token>; HttpOnly                │
│                                                                 │
│  EVERY REQUEST (reads session)                                  │
│  ├─ Browser auto-sends Cookie: session_token=<token>           │
│  ├─ Server reads cookie value                                   │
│  ├─ SELECT FROM sessions WHERE token = <cookie_value>          │
│  ├─ Check: expired? user active?                                │
│  └─ Return user object → route runs                             │
│                                                                 │
│  LOGOUT (destroys session)                                      │
│  ├─ DELETE FROM sessions WHERE token = <cookie_value>          │
│  └─ Set-Cookie: session_token=; Max-Age=0                      │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```

---

# Now: Test it LIVE!

Everything above was simulated in the notebook. Now let's run the **actual FastAPI server** and test through Swagger UI + the React frontend.

### Start the servers:

**Terminal 1 — Backend:**
```bash
cd /Users/datasense/Desktop/agent-flow
python -m uvicorn app:app --reload --port 8000
```

**Terminal 2 — Frontend:**
```bash
cd /Users/datasense/Desktop/agent-flow/frontend
npm run dev
```

### Test in Swagger UI: `http://localhost:8000/docs`

Run through all 9 tests from the testing checklist:
1. POST /auth/signup → create user
2. POST /auth/login (bad email) → 404
3. POST /auth/login (good email) → cookie appears in DevTools
4. GET /auth/me → returns user
5. POST /chat → protected, works with cookie
6. Edit cookie in DevTools → 401
7. POST /auth/logout → cookie disappears
8. Login twice → two sessions
9. Console: `document.cookie` → empty (HttpOnly!)

### Test in Frontend: `http://localhost:5173`

1. Click "Start chatting" → see login form
2. Sign up with email + name
3. See personalized greeting: "Hi Alice, what can I help with?"
4. Send a chat message → works
5. Click "Log out" → back to landing
6. Try "Start chatting" again → login form (session is gone)

---
# What's Next: Phase 2 — Google OAuth

Everything works. But there's a **giant hole** — we never verified the email. Anyone could sign up as `elon@tesla.com` and we'd believe them.

We need someone we **trust** to verify identity. That's where Google comes in.

```
Phase 1 (what we just built):
  User types email → we trust them → create session → set cookie
                     ^^^^^^^^^^^
                     THE PROBLEM

Phase 2 (next):
  User clicks "Login with Google" → Google verifies identity → we get email from Google
                                    ^^^^^^^^^^^^^^^^^^^^^^^^^
                                    GOOGLE is the trusted party
```

**The session and cookie part stays EXACTLY the same.** The only thing that changes is HOW we confirm the user's email:

| | Phase 1 | Phase 2 |
|---|---------|---------|
| Identity verification | We trust the email (insecure) | Google verifies (secure) |
| Session creation | `secrets.token_hex(32)` | Same |
| Cookie setting | `set_cookie(httponly=True)` | Same |
| Session lookup | `query(SessionRecord)` | Same |
| Logout | Delete session + clear cookie | Same |